# L3-30：非 Ascon AXI DMA 回环段

默认使用仓库内已验证打包的非 Ascon DMA 回环工件：

```text
board/vivado_zynq7000_xc7z020_axi_dma_loopback/candidate/
```

环境变量仅用于覆盖默认工件：

```bash
export L3_DMA_DIR=/path/to/non_ascon_overlay
# 或分别设置
export L3_DMA_BIT=/path/to/design.bit
export L3_DMA_MANIFEST=/path/to/manifest.json
```

缺少工件时会保存 `NOT_RUN` 并非零退出。不会使用 BaseOverlay 中未核实回环路径的
`axi_dma_0` 冒充通过。

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import platform
import subprocess
import sys

import pynq


def write_json(path, value):
    Path(path).write_text(
        json.dumps(value, indent=2, ensure_ascii=False, default=str) + "\n",
        encoding="utf-8",
    )


repo = Path("/home/xilinx/jupyter_notebooks/amd_fpga_2026").resolve()
assert (repo / "skill/ab_eval_pynq/test_board").is_dir(), (
    "fixed repository path does not exist: %s" % repo
)
os.chdir(str(repo))
sys.path.insert(0, str(repo))
sys.path.insert(0, str(repo / "skill/pynq-overlay-workflow/scripts"))

run_dir = (
    Path(os.environ["L3_RUN_DIR"]).resolve()  # 可选：不填时读取 l3_current_session.json
    if os.environ.get("L3_RUN_DIR")
    else Path(
        json.loads(
            (repo / "data/skill/l3_current_session.json").read_text(encoding="utf-8")
        )["run_dir"]
    ).resolve()
)
assert run_dir.is_dir(), run_dir
assert (repo / "data/skill").resolve() in run_dir.parents, (
    "L3_RUN_DIR must stay under data/skill"
)
assert (run_dir / "00_preflight.json").is_file(), "run l3_00_preflight.ipynb first"
run_id = run_dir.name


def base_record(case, claim_level):
    return {
        "case": case,
        "status": "RUNNING",
        "claim_level": claim_level,
        "timestamp_local": datetime.now().isoformat(timespec="seconds"),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "run_id": run_id,
        "run_dir": str(run_dir),
        "hostname": platform.node(),
        "python_version": platform.python_version(),
        "pynq_version": pynq.__version__,
        "kernel": platform.platform(),
        "git_commit": subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=str(repo), text=True
        ).strip(),
        "checks": [],
    }


def check(record, name, condition, detail=None):
    item = {"name": name, "status": "PASS" if condition else "FAIL", "detail": detail}
    record["checks"].append(item)
    assert condition, item

In [2]:
import contextlib
import time

import numpy as np
import pynq
from pynq import MMIO
from pynq.ps import Clocks
from pynq_skill import artifacts, dma, golden, hwh, overlay_check

record = base_record("non_ascon_dma", "not_measured")  # 成功后才升级为 hardware_measured
out_path = run_dir / "30_non_ascon_dma.json"
record.update(
    {
        "simulated": False,
        "artifact": None,  # 运行时填写 bit 与 manifest 的绝对路径
        "selected_ip": None,  # 分析 hwh 后自动选择；多个 DMA 时用 L3_DMA_IP 指定
        "transfers": [],  # 运行时追加每种长度的原始结果
        "mutation_checked": False,  # 错误数据比较通过后自动改为 True
        "edge_case_coverage": {
            "odd_length": "pending",
            "back_to_back_same_length": "pending",
            "over_max_rejected_before_start": "pending",
            "short_output": "not_tested: requires a dedicated short-packet PL source",
            "missing_tlast": "not_tested: would intentionally hang the formal DMA path",
        },
    }
)


def fail_not_run(reason):
    record["status"] = "NOT_RUN"
    record["claim_level"] = "not_measured"
    record["reason"] = reason
    write_json(out_path, record)
    raise RuntimeError(reason)


default_dma_dir = (
    repo / "board/vivado_zynq7000_xc7z020_axi_dma_loopback/candidate"
)
directory_env = os.environ.get("L3_DMA_DIR")
bit_value = os.environ.get("L3_DMA_BIT")  # 可选：覆盖默认 .bit
manifest_value = os.environ.get("L3_DMA_MANIFEST")  # 可选：覆盖默认 manifest
if not bit_value:
    directory = directory_env or str(default_dma_dir)
    if directory:
        directory_path = Path(directory).resolve()
        bit_files = sorted(directory_path.glob("*.bit"))
        manifest_files = sorted(directory_path.glob("manifest.json"))
        if len(bit_files) != 1 or len(manifest_files) != 1:
            fail_not_run(
                "L3_DMA_DIR must contain exactly one .bit and one manifest.json"
            )
        bit_value = str(bit_files[0])
        manifest_value = manifest_value or str(manifest_files[0])
if bit_value and not manifest_value:
    manifest_value = str(Path(bit_value).with_name("manifest.json"))
if not bit_value or not manifest_value:
    fail_not_run(
        "non-Ascon DMA loopback is not configured; set L3_DMA_BIT and L3_DMA_MANIFEST"
    )

bit_path = Path(bit_value).resolve()
manifest_path = Path(manifest_value).resolve()
if not bit_path.is_file() or not manifest_path.is_file():
    fail_not_run("configured DMA artifact or manifest is missing")

record["artifact"] = {
    "source": (
        "L3_DMA_BIT" if os.environ.get("L3_DMA_BIT")
        else ("L3_DMA_DIR" if directory_env else "repository_default")
    ),
    "bit": str(bit_path),
    "manifest": str(manifest_path),
}
dma_driver = None  # 仅用于异常路径判断是否已创建驱动；正常情况下保持 None

try:
    manifest_data = json.loads(manifest_path.read_text(encoding="utf-8"))
    manifest_design = manifest_data.get("design", {})
    manifest_dma = manifest_design.get("axi_dma_0", {})
    record["manifest"] = {
        "path": str(manifest_path),
        "sha256": artifacts.sha256_file(manifest_path),
        "part": manifest_data.get("part"),
        "top": manifest_data.get("top"),
        "hardware_validation": manifest_data.get("hardware_validation"),
        "axi_dma_0": manifest_dma,
    }
    check(record, "manifest_part", manifest_data.get("part") == "xc7z020clg400-1")
    check(record, "manifest_dma_vlnv", manifest_dma.get("vlnv") == "xilinx.com:ip:axi_dma:7.1")
    check(record, "manifest_simple_mode", str(manifest_dma.get("c_include_sg")) == "0")
    check(record, "manifest_both_channels", (
        str(manifest_dma.get("c_include_mm2s")) == "1"
        and str(manifest_dma.get("c_include_s2mm")) == "1"
    ))
    check(record, "manifest_stream_loopback_tlast", "TLAST" in str(manifest_dma.get("stream")))
    check(record, "manifest_hp0_window", "HP0" in str(manifest_dma.get("memory")))
    overlay, info, verification = overlay_check.load_overlay(
        bit_path,
        manifest=manifest_path,
    )
    dma_candidates = sorted(
        name
        for name, item in info["ips"].items()
        if str(item.get("vlnv", "")).startswith("xilinx.com:ip:axi_dma:")
    )
    selected = os.environ.get("L3_DMA_IP")
    if selected is None and len(dma_candidates) == 1:
        selected = dma_candidates[0]
    if selected is None:
        fail_not_run("expected one AXI DMA; set L3_DMA_IP explicitly")
    if selected not in overlay.ip_dict:
        raise RuntimeError("DMA %r is in hwh but missing from runtime ip_dict" % selected)
    expected_base = int(str(manifest_dma.get("base")), 0)
    check(record, "dma_base_matches_manifest", (
        overlay.ip_dict[selected].get("phys_addr") == expected_base
    ), {"runtime": overlay.ip_dict[selected].get("phys_addr"), "manifest": expected_base})

    overlay_check.verify_overlay(
        overlay,
        info,
        required={selected: info["ips"][selected]["vlnv"]},
        mmio_factory=lambda address, size: MMIO(address, size),
    )
    check(record, "overlay_loaded", overlay.is_loaded())
    check(record, "dma_ip_present", selected in overlay.ip_dict)

    ip = overlay.ip_dict[selected]
    parameters = ip.get("parameters") or info["ips"][selected]["parameters"]
    dma_driver = dma.BoundedSimpleDMA.from_parameters(
        MMIO(ip["phys_addr"], ip["addr_range"]),
        parameters,
        poll_s=0.0005,
    )
    record["selected_ip"] = selected
    record["verification"] = verification
    record["fclk0_hwh_mhz"] = info["clocks"].get("fclk0_mhz")
    record["fclk0_pynq_register_derived_mhz"] = getattr(
        Clocks(), "fclk0_mhz", None
    )
    record["dma_config"] = {
        "max_transfer": dma_driver.max_transfer,
        "alignment": {hex(key): value for key, value in dma_driver.align.items()},
    }

    # Include 9 bytes for the L2 odd-length failure mode, then repeat 16 bytes back-to-back.
    lengths = [value for value in (8, 9, 16, 64, 256) if value <= dma_driver.max_transfer]
    if not lengths:
        raise RuntimeError("DMA max transfer is too small for a meaningful test")
    maximum = max(lengths)
    tx = pynq.allocate((maximum,), dtype="u1")
    rx = pynq.allocate((maximum,), dtype="u1")
    hp0_limit = 0x40000000  # manifest: PS7 S_AXI_HP0 aperture 0x00000000..0x3FFFFFFF
    check(record, "tx_buffer_in_hp0_window", 0 <= int(tx.physical_address) < hp0_limit)
    check(record, "rx_buffer_in_hp0_window", 0 <= int(rx.physical_address) < hp0_limit)
    previous = None  # 第一次比较没有上一次输出；后续循环自动更新，不需要手填

    sequence = [(str(length), length) for length in lengths]
    if 16 in lengths:
        sequence.append(("16_repeat", 16))

    for case_index, (label, length) in enumerate(sequence):
        payload = bytes(((length + case_index) * 37 + 11) & 0xFF for index in range(length))
        tx[:length] = np.frombuffer(payload, dtype=np.uint8)
        rx[:] = np.frombuffer(b"\xA5" * maximum, dtype=np.uint8)
        received = dma_driver.transfer(
            tx,
            rx,
            timeout_s=max(2.0, length / 4096.0),
            tx_len=length,
            rx_len=length,
        )
        actual = bytes(rx[:received])
        comparison = golden.check(
            actual,
            payload[:received],
            name="dma_loopback_%s" % label,
            previous=previous,
            sentinel=0xA5,
        )
        item = {
            "label": label,
            "requested": length,
            "received": received,
            "tx_physical_address": int(tx.physical_address),
            "rx_physical_address": int(rx.physical_address),
            "comparison": comparison,
        }
        record["transfers"].append(item)
        check(record, "dma_length_%s" % label, received == length, item)
        check(record, "dma_data_%s" % label, comparison["ok"], comparison)
        previous = actual
    record["edge_case_coverage"]["odd_length"] = "pass" if 9 in lengths else "not_available"
    record["edge_case_coverage"]["back_to_back_same_length"] = (
        "pass" if any(label == "16_repeat" for label, _ in sequence) else "not_available"
    )

    # Over-limit requests must be rejected by the Python length gate before DMA starts.
    if dma_driver.max_transfer <= (1 << 20):
        oversized = pynq.allocate((dma_driver.max_transfer + 1,), dtype="u1")
        try:
            dma_driver.transfer(
                oversized,
                oversized,
                timeout_s=2.0,
                tx_len=dma_driver.max_transfer + 1,
                rx_len=dma_driver.max_transfer + 1,
            )
        except ValueError as exc:
            record["over_max_error"] = {"type": type(exc).__name__, "text": str(exc)}
            check(record, "dma_over_max_rejected_before_start", True)
            record["edge_case_coverage"]["over_max_rejected_before_start"] = "pass"
        else:
            check(record, "dma_over_max_rejected_before_start", False)
        finally:
            del oversized
    else:
        record["edge_case_coverage"]["over_max_rejected_before_start"] = (
            "not_tested: max_transfer allocation would exceed 1 MiB"
        )

    last_expected = payload
    mutated = bytes([last_expected[0] ^ 0x01]) + last_expected[1:]
    golden.mutation_must_fail(golden.check, mutated, last_expected)
    record["mutation_checked"] = True
    check(record, "comparator_detects_error", record["mutation_checked"])
    dma_driver.recover(timeout_s=2.0)
    check(record, "dma_recovered_to_halted", all(
        dma_driver.mmio.read(base + dma.SR) & dma.SR_HALTED
        for base in (dma.MM2S, dma.S2MM)
    ))

    record["status"] = "PASS"
    record["claim_level"] = "hardware_measured"
except BaseException as exc:
    if dma_driver is not None:
        try:
            dma_driver.recover(timeout_s=2.0)
            record["recovery_after_failure"] = "PASS"
        except BaseException as recovery_error:
            record["recovery_after_failure"] = {
                "status": "FAIL",
                "error_type": type(recovery_error).__name__,
                "error": str(recovery_error),
            }
    record["status"] = "FAIL" if record.get("status") != "NOT_RUN" else "NOT_RUN"
    record["error_type"] = type(exc).__name__
    record["error"] = str(exc)
    write_json(out_path, record)
    raise
else:
    write_json(out_path, record)
    print(json.dumps(record, indent=2, ensure_ascii=False))
    print("saved", out_path)

{
  "case": "non_ascon_dma",
  "status": "PASS",
  "claim_level": "hardware_measured",
  "timestamp_local": "2026-10-31T06:37:44",
  "timestamp_utc": "2026-10-31T06:37:44+00:00",
  "run_id": "l3_20261031_063709",
  "run_dir": "/home/xilinx/jupyter_notebooks/amd_fpga_2026/data/skill/l3_20261031_063709",
  "hostname": "pynq",
  "python_version": "3.8.2",
  "pynq_version": "2.7.0",
  "kernel": "Linux-5.4.0-xilinx-v2020.2-armv7l-with-glibc2.29",
  "git_commit": "ac9143d5c6acc9e7ac61f907a7e802eb6b35cddd",
  "checks": [
    {
      "name": "manifest_part",
      "status": "PASS",
      "detail": null
    },
    {
      "name": "manifest_dma_vlnv",
      "status": "PASS",
      "detail": null
    },
    {
      "name": "manifest_simple_mode",
      "status": "PASS",
      "detail": null
    },
    {
      "name": "manifest_both_channels",
      "status": "PASS",
      "detail": null
    },
    {
      "name": "manifest_stream_loopback_tlast",
      "status": "PASS",
      "detail": null
    }